# Probar el extractor de características por canal

Este notebook ejecuta el banco de filtros, LSTM bidireccional y atención de época de PhysioEx 2.0.2. Procesa dos archivos y conserva dos épocas por archivo. No entrena ni modifica prototipos o clasificación.

La extracción con pesos publicados y la prueba con pesos aleatorios son modos distintos. Por defecto intenta usar el checkpoint oficial; si no es accesible, puedes elegir explícitamente el modo `random` para comprobar la ejecución, sin afirmar que sus características estén aprendidas.

No filtra ni normaliza tus señales. Usa una conversión configurable V → µV. Aunque se carguen pesos aprendidos, la transferencia a tus señales sin filtrado todavía necesita validación: el preset de referencia también filtra.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Selección pequeña

Se leen dos archivos y se conservan solo dos épocas de cada uno. Un NPZ comprimido necesita descomprimir la matriz X de cada archivo seleccionado: el pico de memoria todavía incluye un registro completo, aunque no se conserva todo el conjunto. Puedes elegir los nombres explícitamente. No se forman secuencias entre épocas.


In [ ]:
from pathlib import Path
import numpy as np

DATA_DIR = Path('/content/drive/MyDrive/SleepCassette_procesado')
FILE_NAMES = []  # Opcional: dos nombres .npz distintos
INPUT_UNITS = 'V'  # 'V' si son voltios; 'uV' si ya son microvoltios
EPOCHS_PER_FILE = 2
assert INPUT_UNITS in ('V', 'uV')
assert DATA_DIR.is_dir(), f'No existe {DATA_DIR}'
if FILE_NAMES:
    assert len(FILE_NAMES) == 2 and len(set(FILE_NAMES)) == 2
    assert all(Path(n).name == n and n.endswith('.npz') for n in FILE_NAMES)
    selected = sorted(DATA_DIR / n for n in FILE_NAMES)
else:
    selected = sorted(DATA_DIR.glob('*.npz'))[:2]
assert len(selected) == 2 and all(p.is_file() for p in selected)
print('Seleccionados:', [p.name for p in selected])


In [ ]:
import gc

signals, labels, subjects, sample_ids = [], [], [], []
for path in selected:
    with np.load(path, allow_pickle=False) as data:
        x, y, subj = data['X'], data['y'], data['subj']
        assert x.ndim == 3 and x.shape[1:] == (3, 3000) and len(x) > 0
        assert y.shape == (len(x),) and np.issubdtype(y.dtype, np.integer)
        assert np.isin(y, np.arange(1, 7)).all()
        n = min(EPOCHS_PER_FILE, len(x))
        if subj.size == 1:
            ids = np.repeat(subj.reshape(-1), n).astype(str)
        else:
            assert subj.shape == (len(x),)
            ids = subj[:n].astype(str)
        signals.append(x[:n].astype(np.float32, copy=True))
        labels.append(y[:n].astype(np.int64) - 1)
        subjects.extend(ids.tolist())
        sample_ids.extend(f'{path.name}:{i}' for i in range(n))
    del x, y, subj
    gc.collect()
x_original = np.concatenate(signals)
y_batch = np.concatenate(labels)
del signals, labels
assert np.isfinite(x_original).all()
# Cambio de unidades explícito; no modifica los archivos ni normaliza por época.
x_uv = x_original * (1e6 if INPUT_UNITS == 'V' else 1.0)
assert np.isfinite(x_uv).all()
print('Lote:', x_uv.shape, '| Etiquetas 0–5:', y_batch)
print('Sujetos:', subjects)
print('Percentiles de amplitud absoluta por canal (microvoltios):')
print(np.percentile(np.abs(x_uv), [50, 95, 99], axis=(0, 2)))


## Espectrograma por canal

Se sigue el cálculo de `XSleepNetSpectrogram` de PhysioEx 2.0.2: ventana Hamming periódica de 200 muestras, solapamiento 100, FFT 256 y `10*log10(PSD + eps_float32)`. Se explicitan los valores por defecto de SciPy: detrend constante y escala de densidad. No se aplica clipping. Se conserva el espectrograma logarítmico sin z-score adicional.

Esto reproduce la operación espectral sobre la señal suministrada; no el preset completo `seqsleepnet`, que también filtra. La escala es dB respecto a 1 µV²/Hz, con el epsilon numérico indicado. No se puede comparar directamente con espectros calculados en voltios sin revisar escala y epsilon.


In [ ]:
from scipy.signal import spectrogram, get_window

def epoch_spectrogram(x):
    """(B, 3, 3000), en microvoltios → (B, 3, 29, 129)."""
    x = np.asarray(x, dtype=np.float64)
    assert x.ndim == 3 and x.shape[1:] == (3, 3000)
    frequencies, times, power = spectrogram(
        x, fs=100.0, window=get_window('hamming', 200),
        nperseg=200, noverlap=100, nfft=256,
        detrend='constant', scaling='density', mode='psd', axis=-1,
    )
    log_power = 10.0 * np.log10(power + np.finfo(np.float32).eps)
    specs = log_power.swapaxes(-1, -2).astype(np.float32)
    assert specs.shape == (len(x), 3, 29, 129)
    assert np.isfinite(specs).all()
    return specs, frequencies, times

specs, frequencies, times = epoch_spectrogram(x_uv)
# Cada canal será un ejemplo independiente para el mismo encoder compartido.
encoder_input = specs.reshape(-1, 1, 29, 129)
np.testing.assert_array_equal(encoder_input[0, 0], specs[0, 0])
np.testing.assert_array_equal(encoder_input[2, 0], specs[0, 2])
print('Espectrogramas:', specs.shape)
print('Entrada futura del encoder:', encoder_input.shape)
print('Frecuencias:', frequencies[0], 'a', frequencies[-1], 'Hz')


In [ ]:
import matplotlib.pyplot as plt

channel_names = ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal']
time_signal = np.arange(3000) / 100.0
fig, axes = plt.subplots(3, 2, figsize=(13, 9), constrained_layout=True)
for c, name in enumerate(channel_names):
    axes[c, 0].plot(time_signal, x_uv[0, c], linewidth=0.6)
    axes[c, 0].set(title=name, xlabel='Tiempo (s)', ylabel='µV')
    image = axes[c, 1].pcolormesh(times, frequencies, specs[0, c].T, shading='auto')
    axes[c, 1].set(title=f'Espectrograma: {name}', xlabel='Tiempo (s)', ylabel='Frecuencia (Hz)')
    fig.colorbar(image, ax=axes[c, 1], label='dB re 1 µV²/Hz (con epsilon)')
plt.show()


## Encoder real, sin mezclar canales

Se incluyen las clases para que no tengas que instalar PhysioEx ni subir archivos Python. Las operaciones provienen de PhysioEx v2.0.2 (MIT); se expone su clase interna `_SeqEpochEncoder` como `SeqEpochEncoder`. El banco de filtros, la LSTM y la atención conservan su estructura. No se incluye ChannelMixer ni el encoder de contexto entre épocas.

Solo PyTorch se usa en esta parte. Puedes activar GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución**; también funciona en CPU.


In [ ]:
"""Per-channel SeqSleepNet epoch encoder, adapted from PhysioEx v2.0.2.

Sources (MIT license):
https://github.com/guidogagl/physioex/blob/v2.0.2/physioex/models/seqsleepnet.py
https://github.com/guidogagl/physioex/blob/v2.0.2/physioex/models/protosleepnet.py
Copyright (c) PhysioEx contributors. See LICENSE_PHYSIOEX alongside this file.

The filterbank/LSTM/attention operations are preserved. The nested upstream
encoder is exposed as a standalone class with explicit constructor defaults.
No ChannelMixer, inter-epoch encoder, prototypes or classifier are included.
"""
import torch
from torch import nn

class LearnableFilterbank(nn.Module):
    def __init__(
        self,
        in_chan: int = 2,
        F: int = 129,
        nfilt: int = 32,
        nfft: int = 256,
        sf: int = 100,
        lowfreq: int = 0,
        highfreq: int = 50,
    ):
        super().__init__()
        self.F, self.D = F, nfilt

        S = torch.zeros((in_chan, F, nfilt), dtype=torch.float32)

        for i in range(in_chan):
            S[i] = self.lin_tri_filter_shape(nfilt, nfft, sf, lowfreq, highfreq)

        W = torch.zeros((in_chan, F, nfilt), dtype=torch.float32)

        self.W = nn.Parameter(W, requires_grad=True)
        self.S = nn.Parameter(S, requires_grad=False)

        nn.init.normal_(self.W)

    def forward(self, x):
        Wfb = torch.mul(torch.sigmoid(self.W), self.S)

        return torch.matmul(x, Wfb)

    def lin_tri_filter_shape(
        self, nfilt=20, nfft=512, sf=16000, lowfreq=0, highfreq=None
    ):
        highfreq = highfreq or sf / 2
        assert highfreq <= sf / 2, "highfreq is greater than sf/2"

        hzpoints = torch.linspace(lowfreq, highfreq, nfilt + 2)
        bin = torch.floor((nfft + 1) * hzpoints / sf)

        fbank = torch.zeros([nfilt, nfft // 2 + 1])
        for j in range(0, nfilt):
            for i in range(int(bin[j]), int(bin[j + 1])):
                fbank[j, i] = (i - bin[j]) / (bin[j + 1] - bin[j])
            for i in range(int(bin[j + 1]), int(bin[j + 2])):
                fbank[j, i] = (bin[j + 2] - i) / (bin[j + 2] - bin[j + 1])
        fbank = torch.transpose(fbank, 0, 1)
        return fbank.float()

class AttentionLayer(nn.Module):
    def __init__(
        self,
        hidden_size,
        attention_size: int = 32,
        time_major: bool = False,
    ):
        super().__init__()

        W_omega = torch.zeros((hidden_size, attention_size), dtype=torch.float32)
        b_omega = torch.zeros((attention_size), dtype=torch.float32)
        u_omega = torch.zeros((attention_size), dtype=torch.float32)

        self.W_omega = nn.Parameter(W_omega)
        self.b_omega = nn.Parameter(b_omega)
        self.u_omega = nn.Parameter(u_omega)

        nn.init.normal_(self.W_omega, std=0.1)
        nn.init.normal_(self.b_omega, std=0.1)
        nn.init.normal_(self.u_omega, std=0.1)

    def forward(self, x, r_alphas=False):
        batch_size, sequence_length, hidden_size = x.size()

        v = torch.tanh(
            torch.matmul(
                torch.reshape(x, [batch_size * sequence_length, hidden_size]),
                self.W_omega,
            )
            + torch.reshape(self.b_omega, [1, -1])
        )
        vu = torch.matmul(v, torch.reshape(self.u_omega, [-1, 1]))
        exps = torch.reshape(torch.exp(vu), [-1, sequence_length])
        alphas = exps / torch.reshape(torch.sum(exps, 1), [-1, 1])

        output = torch.sum(
            x * torch.reshape(alphas, [batch_size, sequence_length, 1]), 1
        )
        if r_alphas:
            return output, alphas
        return output

class SeqEpochEncoder(nn.Module):
            def __init__(self, F=129, D=32, nfft=256, fs=100, lowfreq=0, highfreq=50,
                 seqnhidden1=64, seqnlayer1=4, attentionsize=32):
                super().__init__()
                self.filterbank = LearnableFilterbank(
                    in_chan=1, F=F, nfilt=D, nfft=nfft, sf=fs,
                    lowfreq=lowfreq, highfreq=highfreq,
                )
                self.lstm = nn.LSTM(
                    D, seqnhidden1, num_layers=seqnlayer1,
                    batch_first=True, bidirectional=True,
                )
                self.attention = AttentionLayer(2 * seqnhidden1, attentionsize)

            def forward(self, x):
                x = self.filterbank(x)
                x = x.permute(0, 2, 1, 3)
                x = x.reshape(x.shape[0], x.shape[1], -1)
                x, _ = self.lstm(x)
                x = self.attention(x)
                return x

def encode_channels(encoder, spectrograms):
    """(B, 3, 29, 129) -> (B, 128, 1, 3), with shared independent encoding.

    Pass an encoder in eval mode for reproducible inference. No normalization,
    sensor weighting or channel mixing is performed here.
    """
    if spectrograms.ndim != 4 or tuple(spectrograms.shape[1:]) != (3, 29, 129):
        raise ValueError("Expected spectrograms with shape (B, 3, 29, 129)")
    batch = spectrograms.shape[0]
    h = encoder(spectrograms.reshape(batch * 3, 1, 29, 129))
    return h.reshape(batch, 3, -1).permute(0, 2, 1).unsqueeze(2).contiguous()


def load_epoch_encoder_weights(encoder, checkpoint):
    """Load only encoder weights from an upstream state_dict, strictly.

    Supports a full model's epoch_encoder.* keys or a standalone encoder dict.
    Does not accept pickled model objects or silently skip incompatible layers.
    """
    if not isinstance(checkpoint, dict):
        raise ValueError("Expected a state_dict checkpoint")
    state = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    if not isinstance(state, dict):
        raise ValueError("Expected a state_dict mapping")
    selected = {k[len("epoch_encoder."):]: v for k, v in state.items()
                if k.startswith("epoch_encoder.")}
    if not selected:
        selected = state
    encoder.load_state_dict(selected, strict=True)
    return encoder


## Elegir y cargar pesos

- `pretrained`: descarga `model.pt` del modelo oficial `4rooms/sleep-prototypes/protosleepnet-gagliardi`, fijando la revisión devuelta por Hugging Face. Solo carga `epoch_encoder.*`, con coincidencia estricta.
- `local`: utiliza un checkpoint state_dict ya descargado en Colab.
- `random`: prueba mecánica reproducible; los vectores no son características entrenadas.

Si la descarga falla, no se cambia de modo automáticamente. Puedes descargar los pesos desde el enlace oficial y usar `local`, o seleccionar `random` para comprobar el flujo. Un checkpoint con otra arquitectura fallará explícitamente. No se cargan objetos Python completos: se usa `weights_only=True`.


In [ ]:
import json
import urllib.request
import hashlib
import torch

WEIGHTS_MODE = 'pretrained'  # 'pretrained', 'local' o 'random'
LOCAL_CHECKPOINT = '/content/model.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
encoder = SeqEpochEncoder()
checkpoint_revision = None
checkpoint_sha256 = None

if WEIGHTS_MODE == 'pretrained':
    repo = '4rooms/sleep-prototypes'
    with urllib.request.urlopen(f'https://huggingface.co/api/models/{repo}', timeout=60) as response:
        info = json.load(response)
    checkpoint_revision = info['sha']
    model_file = 'protosleepnet-gagliardi/model.pt'
    assert model_file in {f['rfilename'] for f in info['siblings']}, (
        'La publicación cambió: revisa los archivos del repositorio oficial antes de continuar.'
    )
    checkpoint_path = Path('/content/protosleepnet-gagliardi-model.pt')
    url = f'https://huggingface.co/{repo}/resolve/{checkpoint_revision}/{model_file}'
    urllib.request.urlretrieve(url, checkpoint_path)
    print('Checkpoint oficial:', url)
elif WEIGHTS_MODE == 'local':
    checkpoint_path = Path(LOCAL_CHECKPOINT)
    assert checkpoint_path.is_file(), f'No existe {checkpoint_path}'
elif WEIGHTS_MODE != 'random':
    raise ValueError('Modo desconocido')

if WEIGHTS_MODE != 'random':
    digest = hashlib.sha256()
    with checkpoint_path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    checkpoint_sha256 = digest.hexdigest()
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    load_epoch_encoder_weights(encoder, checkpoint)
    del checkpoint
    print('Pesos del encoder cargados con strict=True. SHA256:', checkpoint_sha256)
else:
    print('PRUEBA CON PESOS ALEATORIOS: no demuestra extracción aprendida ni precisión.')

encoder = encoder.to(DEVICE).eval()
print('Dispositivo:', DEVICE, '| Parámetros:', sum(p.numel() for p in encoder.parameters()))


## Ejecutar el extractor y comprobar la independencia de canales

In [ ]:
import time

spec_tensor = torch.from_numpy(specs).to(DEVICE)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
start = time.perf_counter()
with torch.inference_mode():
    features = encode_channels(encoder, spec_tensor)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start
assert features.shape == (len(specs), 128, 1, 3)
assert torch.isfinite(features).all()

# Modificar un canal debe dejar intactos los otros dos.
with torch.inference_mode():
    changed = spec_tensor.clone()
    changed[:, 0] += 1.0
    changed_features = encode_channels(encoder, changed)
torch.testing.assert_close(features[..., 1:], changed_features[..., 1:])
print('Entrada:', tuple(spec_tensor.shape))
print('Salida:', tuple(features.shape), '| Tiempo del lote (incluye primera ejecución):', round(elapsed, 3), 's')
print('Canales independientes: comprobado')
print('Primeras 8 características de la primera época (filas = canales):')
print(features[0, :8, 0, :].T.cpu().numpy())
print('Norma de cada vector, por época y canal:')
print(features[:, :, 0, :].norm(dim=1).cpu().numpy())


## Ver los vectores y sus similitudes

Cada fila del mapa es una época; cada columna es una característica aprendida. Los colores no son probabilidades ni etiquetas de sueño. La matriz de cosenos compara ejemplos dentro del mismo canal: es una comprobación exploratoria, no una medición de precisión ni de separabilidad de etapas. Con pocos ejemplos no se puede concluir rendimiento.


In [ ]:
import torch.nn.functional as F

vectors = features[:, :, 0, :].detach().cpu()
fig, axes = plt.subplots(3, 2, figsize=(14, 9), constrained_layout=True)
for c, name in enumerate(channel_names):
    image = axes[c, 0].imshow(vectors[:, :, c].numpy(), aspect='auto', cmap='coolwarm')
    axes[c, 0].set(title=f'{name}: 128 características', xlabel='Característica', ylabel='Época del lote')
    fig.colorbar(image, ax=axes[c, 0])
    normalized = F.normalize(vectors[:, :, c], dim=1)
    cosine = normalized @ normalized.T
    image = axes[c, 1].imshow(cosine.numpy(), vmin=-1, vmax=1, cmap='coolwarm')
    axes[c, 1].set(title=f'{name}: similitud coseno', xlabel='Época', ylabel='Época')
    fig.colorbar(image, ax=axes[c, 1])
plt.show()
print('Modo de pesos:', WEIGHTS_MODE)
print('Etiquetas originales convertidas:', y_batch.tolist())


## Guardar el resultado pequeño en Colab

Se guarda únicamente el lote probado, sus sujetos y etiquetas. No se modifica Drive. `features` tiene `(B, 128, 1, 3)`. El archivo incluye modo de pesos, unidades y revisión para no confundir pruebas aleatorias con extracción preentrenada.


In [ ]:
OUTPUT_PATH = Path('/content/seqepoch_features_prueba.npz')
np.savez_compressed(
    OUTPUT_PATH, features=features.detach().cpu().numpy(), y=y_batch,
    subj=np.asarray(subjects), sample_id=np.asarray(sample_ids),
    channel_names=np.asarray(channel_names), weights_mode=WEIGHTS_MODE,
    input_units=INPUT_UNITS, spectral_units='uV',
    checkpoint_revision=checkpoint_revision or '', checkpoint_sha256=checkpoint_sha256 or '',
    physioex_version='2.0.2', fs=100, nperseg=200, noverlap=100, nfft=256,
    filtering='none_added', normalization='none_added',
)
print('Guardado:', OUTPUT_PATH)


## Qué demuestra esta prueba

- Ejecuta el encoder de época real (banco de filtros aprendible → LSTM bidireccional → atención).
- Produce 128 características separadas para cada uno de los tres canales.
- Si se cargaron pesos publicados, utiliza sus parámetros aprendidos, pero no demuestra que sean adecuados para estas señales sin filtrado ni para seis categorías.
- En modo aleatorio solo valida estructura y ejecución.
- Todavía no calcula las ramas explícitas de rango/varianza/FFT de ProtoEEG, pesos por canal, prototipos ni predicciones.

Fuentes: [PhysioEx v2.0.2](https://github.com/guidogagl/physioex/tree/v2.0.2), [ProtoSleepNet](https://github.com/guidogagl/protosleepnet), [pesos publicados](https://huggingface.co/4rooms/sleep-prototypes).

### Licencia del código reutilizado (PhysioEx)

```text
MIT License

Copyright (c) 2023 Guido Gagliardi

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

```
